# HW3 sentiment classification

The selected preset supplies three BERT-base members. Equal-weight soft voting averages their class probabilities. Evaluation uses the frozen split. The final fit uses all three labeled partitions to produce submission predictions.

In [ ]:
%%bash
set -e
cd /content

if [ -d Intro_to_AI_HW/.git ]; then
    git -C Intro_to_AI_HW pull --ff-only origin main
else
    git clone --branch main https://github.com/billlllllllly/Intro_to_AI_HW.git
fi

cd Intro_to_AI_HW/IAI_HW3
pip install -r requirements.txt

In [ ]:
%cd /content/Intro_to_AI_HW/IAI_HW3
import re
import pandas as pd

tweets = pd.read_csv("dataset/dataset.csv", keep_default_na=False)["text"]
patterns = {"emoticon": r"(?<!\w)(?:[:=;]-?[()\[\]DP/]|<3|\^_?\^|-_-)(?!\w)",
            "hashtag": r"#\w+", "elongated": r"([A-Za-z])\1{2,}", "HTML entity": r"&\w+;"}
for name, pattern in patterns.items():
    count = tweets.map(re.compile(pattern).search).notna().sum()
    print(f"{name}: {count}/{len(tweets)} ({count/len(tweets):.2%})")


In [ ]:
%cd /content/Intro_to_AI_HW/IAI_HW3
import torch
from main import run

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before training.")

predictions, histories = run(
    ensemble=True, preset="diverse", select="last", epochs=3, final_fit=True,
    checkpoint_dir="/content/hw3_ckpt", run_dir="/content/runs/diverse_v1",
    review_path="/content/neutral_errors.csv",
)

In [ ]:
import pandas as pd

errors = pd.read_csv('/content/neutral_errors.csv')
display(errors.groupby(['label', 'ensemble']).size().rename('count').to_frame())
display(errors.groupby('label', group_keys=False).head(10))

In [ ]:
!python diagnose_benchmark.py /content/runs/diverse_v1

In [ ]:
from pathlib import Path

e0_run = Path("/content/runs/E0")
if (e0_run / "probs.npz").is_file():
    !python compare_runs.py /content/runs/E0 /content/runs/diverse_v1
else:
    print("Save the E0 run in /content/runs/E0 to compare it here.")

In [ ]:
from google.colab import files

output_path = "/content/predictions.csv"
predictions.to_csv(output_path, index=False)
files.download(output_path)

In [ ]:
from shutil import make_archive
from google.colab import files

archive_path = make_archive(
    "/content/diverse_v1", "zip", root_dir="/content/runs", base_dir="diverse_v1",
)
files.download(archive_path)

In [ ]:
import matplotlib.pyplot as plt

for name, history in histories.items():
    epochs = range(1, len(history) + 1)
    figure, axes = plt.subplots(1, 2, figsize=(9, 4))
    figure.suptitle(name)
    for index, title in enumerate(("Cross-entropy", "Macro-F1")):
        axes[index].plot(epochs, [train[index] for train, _ in history], label="Train")
        axes[index].plot(epochs, [val[index] for _, val in history], label="Validation")
        axes[index].set(xlabel="Epoch", title=title, xticks=list(epochs))
        axes[index].legend()
    figure.tight_layout()
    plt.show()